# LoRA fine-tuning: customer email intent (Colab)

Runs `lora-email-intent` end to end on a free Colab GPU: train a LoRA adapter on `Qwen/Qwen2.5-0.5B-Instruct`, then compare
the **untuned** model with the **tuned** one on a held-out test set (36 emails).

**Before you start:** Runtime → Change runtime type → **T4 GPU**. Expect a few minutes of training.
Outputs are cleared; results from the documented run are in the README.

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

In [ ]:
!pip uninstall -y -q torchao  # Colab ships a torchao that PEFT rejects
!pip install -q -U transformers peft accelerate
!git clone -q https://github.com/yassineldieb2-hrd/lora-email-intent.git
%cd lora-email-intent
!pip install -q -e .

## 1. Look at the data and the split

In [ ]:
from lora_email_intent.data import load_jsonl, split
from lora_email_intent.prompts import build_prompt
rows = load_jsonl("data/emails.jsonl")
train, val, test = split(rows)
print(len(train), len(val), len(test))
print(build_prompt(test[0]["email"]), "->", test[0]["intent"])

## 2. Train the LoRA adapter
Only the label tokens contribute to the loss (`train_on_inputs=False`).

In [ ]:
!python -m lora_email_intent.train --epochs 5 --out outputs/lora

## 3. Evaluate: base model (zero-shot) vs LoRA-tuned
The *strict* score requires the answer to be exactly one valid label; the *lenient* score accepts a label mentioned anywhere in the first 200 characters so the untuned model is not penalised only for formatting.

In [ ]:
!python -m lora_email_intent.evaluate --adapter outputs/lora --results outputs/eval_results.json

## 4. Inspect mistakes

In [ ]:
import json
r = json.load(open("outputs/eval_results.json"))
for name in ("base_zero_shot", "lora_tuned"):
    wrong = [e for e in r[name]["examples"] if e["generated"].strip().split("\n")[0].strip().lower() != e["gold"]]
    print(f"{name}: {len(wrong)} of {len(r[name]['examples'])} not an exact match, e.g.")
    for e in wrong[:5]:
        print("  gold =", e["gold"], "| generated =", repr(e["generated"]))

## 5. Download the results
Copy the numbers from `outputs/eval_results.json` into the README results table if you re-run with other settings.


In [ ]:
!zip -qr lora_adapter_and_results.zip outputs
from google.colab import files
files.download("lora_adapter_and_results.zip")